In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import MinMaxScaler
from utils.paths import DATASETS_CLEANED_DIR, DATASETS_PREPROCESSED_DIR

In [2]:
datasets = []

for file in os.listdir(DATASETS_CLEANED_DIR):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(DATASETS_CLEANED_DIR, file))
        datasets.append(df)

In [3]:
# converting degrees to sin and cos for wind direction
heights = [10, 100]

for i, df in enumerate(datasets):
    for h in heights:
        col_name = f"wind_direction_{h}m"

        radians = np.radians(df[col_name])
        df[f"wind_sin_{h}m"] = np.sin(radians)
        df[f"wind_cos_{h}m"] = np.cos(radians)

        df.drop(columns=[col_name], inplace=True)

    datasets[i] = df

In [4]:
# converting from Fahrenheit to Celsius
for i, df in enumerate(datasets):
    df['temperature_2m'] = (df['temperature_2m'] - 32) * 5/9

    datasets[i] = df

In [5]:
# finding v_cutin and v_rated values

v_cutin_list = []
v_rated_list = []

n_percentile = 1

for i, df in enumerate(datasets):
    working_turbines_map = df[df['power'] > 0]

    v_cutin = np.percentile(working_turbines_map['wind_speed_100m'], n_percentile)
    v_rated = np.percentile(working_turbines_map['wind_speed_100m'], 100 - n_percentile)

    v_cutin_list.append(v_cutin)
    v_rated_list.append(v_rated)

    print(f"Dataset {i+1}: v_cutin = {v_cutin:.2f} m/s, v_rated = {v_rated:.2f} m/s")

Dataset 1: v_cutin = 3.10 m/s, v_rated = 13.52 m/s
Dataset 2: v_cutin = 3.14 m/s, v_rated = 16.79 m/s
Dataset 3: v_cutin = 3.12 m/s, v_rated = 14.82 m/s
Dataset 4: v_cutin = 3.11 m/s, v_rated = 14.03 m/s


In [6]:
# normalizing wind_speed columns

heights = [10, 100]
scaler = MinMaxScaler()

for i, df in enumerate(datasets):

    scaler.data_min_ = np.array([v_cutin_list[i]])
    scaler.data_max_ = np.array([v_rated_list[i]])
    scaler.data_range_ = scaler.data_max_ - scaler.data_min_
    scaler.scale_ = (1.0 - 0.0) / scaler.data_range_
    scaler.min_ = 0.0 - scaler.data_min_ * scaler.scale_

    for h in heights:
        v_norm = scaler.transform([df[f'wind_speed_{h}m']])
        v_norm = np.clip(v_norm, 0, 1)  # Ensure values are within [0, 1]
        
        df[f'wind_speed_{h}m'] = np.transpose(v_norm)

    datasets[i] = df

In [7]:
# correcting still invalid records, where wind speed is 0 but power is greater than 0
for i, df in enumerate(datasets):
    df.loc[df['wind_speed_100m'] == 0, 'power'] = 0
    datasets[i] = df

In [8]:
for i, df in enumerate(datasets):
    print(f"Location_{i+1}:")
    print(df[(df['wind_speed_100m'] == 0) & (df['power'] > 0)]['power'])
    print(df[(df['wind_speed_100m'] == 0) & (df['power'] > 0)]['power'].max())
    print(f"\n{'#' * 40}\n")

Location_1:
Series([], Name: power, dtype: float64)
nan

########################################

Location_2:
Series([], Name: power, dtype: float64)
nan

########################################

Location_3:
Series([], Name: power, dtype: float64)
nan

########################################

Location_4:
Series([], Name: power, dtype: float64)
nan

########################################



In [ ]:
# splitting into train, validation and test sets

train_sets = []
test_sets = []
split_ratios = [0.8, 0.2]  # 4 years for training, 1 year for test

for df in datasets:
    n = len(df)
    train_end = int(split_ratios[0] * n)


    train_df = df.iloc[:train_end]
    test_df = df.iloc[train_end:]

    train_sets.append(train_df)
    test_sets.append(test_df)

In [10]:
# scaling columns

cols_to_scale = ['temperature_2m', 'dew_point_2m', 'wind_gusts_10m', 'relative_humidity_2m']

for i in range(len(train_sets)):
    scaler = MinMaxScaler()

    train_sets[i][cols_to_scale] = scaler.fit_transform(train_sets[i][cols_to_scale])
    test_sets[i][cols_to_scale] = scaler.transform(test_sets[i][cols_to_scale])

In [11]:
# calculating the empirical curve

def apply_empirical_curve(df, curve):
    df['v_bin'] = df['wind_speed_100m'].round(2)
    df['p_base'] = df['v_bin'].map(curve)

    return df

for i in range(len(train_sets)):
    train_sets[i]['v_bin'] = train_sets[i]['wind_speed_100m'].round(2)
    empirical_curve = train_sets[i].groupby('v_bin')['power'].mean()
    
    train_sets[i] = apply_empirical_curve(train_sets[i], empirical_curve)
    test_sets[i] = apply_empirical_curve(test_sets[i], empirical_curve)

In [12]:
# calculating the delta_p

for i in range(len(train_sets)):
    train_sets[i]['delta_p'] = train_sets[i]['power'] - train_sets[i]['p_base']
    test_sets[i]['delta_p'] = test_sets[i]['power'] - test_sets[i]['p_base']

In [13]:
# saving the preprocessed datasets

for df in train_sets + test_sets:
    df.drop(columns=['v_bin', 'power',], inplace=True)

os.makedirs(DATASETS_PREPROCESSED_DIR, exist_ok=True)

for i in range(len(train_sets)):
    train_sets[i].to_csv(os.path.join(DATASETS_PREPROCESSED_DIR, f"Location_{i+1}_train.csv"), index=False)
    test_sets[i].to_csv(os.path.join(DATASETS_PREPROCESSED_DIR, f"Location_{i+1}_test.csv"), index=False)

In [14]:
# sanity check

for file in os.listdir(DATASETS_PREPROCESSED_DIR):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(DATASETS_PREPROCESSED_DIR, file))
        
        print(f"{file}:")
        print(df.describe())
        print(df.info())
        print(df.isna().sum())
        print("#" * 40)

Location_2_test.csv:
       temperature_2m  relative_humidity_2m  dew_point_2m  wind_speed_10m  \
count     8760.000000           8760.000000   8760.000000     8760.000000   
mean         0.624618              0.551308      0.611480        0.119737   
std          0.190203              0.221884      0.187315        0.133460   
min          0.072698             -0.012048      0.039007        0.000000   
25%          0.494346              0.385542      0.500000        0.000000   
50%          0.623586              0.554217      0.605496        0.084249   
75%          0.781099              0.713855      0.773271        0.194872   
max          1.022617              1.000000      0.970745        0.800000   

       wind_speed_100m  wind_gusts_10m  wind_sin_10m  wind_cos_10m  \
count      8760.000000     8760.000000   8760.000000   8760.000000   
mean          0.315734        0.279619     -0.169206     -0.021147   
std           0.224030        0.130677      0.657746      0.733760   
min  